# 1.5. Đánh giá hiệu năng thu thập dữ liệu trên Google Colab CPU-only

## 1. Môi trường đo và 7 chỉ số hiệu năng bắt buộc của Milestone 1

Toàn bộ 6 phiên thu thập của 5 thành viên (gồm 4 phiên chạy trọn vẹn 5.000 bài và cặp phiên chạy gốc 4.425 bài kèm phiên `--resume` hoàn tất 5.000 bài của thành viên TRI) đều thực thi trên máy ảo Google Colab Standard CPU-only (`is_google_colab = True`, `cpu_only = True`, 2 vCPU, tổng RAM 12.975,52 MiB). Bộ đo `ResourceProfiler` lấy mẫu cây tiến trình mỗi 0,5 giây (kèm đối chiếu `ru_maxrss` từ nhân Linux) và ghi nhận dung lượng đĩa mỗi 30 giây.

```{note}
**Quy ước đơn vị bộ nhớ nhị phân `MiB` và `GiB`:**
Trong `src/crawler/profiler.py` (`BYTES_PER_MB = 1024**2 = 1.048.576`), mọi thông số bộ nhớ đều được chia theo cơ số nhị phân $1024^2$ byte (`MiB`). Các bảng dưới đây trình bày thống nhất theo đơn vị `MiB`, `GiB` và `byte`.
```

In [ ]:
import html
from pathlib import Path
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch, Rectangle
from matplotlib.ticker import FuncFormatter
from IPython.display import display, Markdown, HTML

data = json.loads((Path.cwd().parent / "report_summary.json").read_text(encoding="utf-8"))
scope, m1, m2, bench = data["scope"], data["m1"], data["m2"], data["m2_benchmark"]

# Bảng màu học thuật tối giản: 1 tông xanh đậm chủ đạo kết hợp thang xám điềm đạm
BOOK_PALETTE = {
    "navy": "#1B365D",
    "steel": "#3B6E8C",
    "slate": "#64748B",
    "muted": "#94A3B8",
    "light": "#CBD5E1",
    "accent": "#2C5282",
    "grid": "#CBD5E1",
}
MEMBER_STYLES = {
    "DAT": ("#1B365D", "-"),
    "TUAN": ("#3B6E8C", "--"),
    "TAN": ("#4A5568", "-."),
    "TRI": ("#64748B", ":"),
    "TRIET": ("#94A3B8", "-"),
}

plt.rcParams.update({
    "font.family": "sans-serif",
    "font.sans-serif": ["Liberation Sans", "DejaVu Sans", "Arial"],
    "figure.figsize": (9.6, 4.0),
    "figure.dpi": 170,
    "savefig.dpi": 170,
    "savefig.bbox": "tight",
    "figure.facecolor": "#FFFFFF",
    "axes.facecolor": "#FFFFFF",
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.edgecolor": "#94A3B8",
    "axes.linewidth": 0.85,
    "axes.titlesize": 10.5,
    "axes.titleweight": "bold",
    "axes.titlecolor": "#1B365D",
    "axes.labelsize": 9.2,
    "axes.labelcolor": "#1E293B",
    "xtick.labelsize": 8.6,
    "ytick.labelsize": 8.6,
    "xtick.color": "#334155",
    "ytick.color": "#334155",
    "grid.color": "#CBD5E1",
    "grid.linestyle": ":",
    "grid.alpha": 0.50,
})


def fmt_int(x) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{int(round(float(x))):,}".replace(",", ".")


def fmt_num(x, digits: int = 2) -> str:
    if x is None or pd.isna(x):
        return "-"
    return f"{float(x):,.{digits}f}".replace(",", "X").replace(".", ",").replace("X", ".")


def fmt_pct(x, digits: int = 2, scale100: bool = False) -> str:
    if x is None or pd.isna(x):
        return "-"
    val = float(x) * (100.0 if scale100 else 1.0)
    return f"{val:.{digits}f}%".replace(".", ",")


def style_table(
    df: pd.DataFrame,
    caption: str | None = None,
    precision: int = 2,
    int_cols: list[str] | None = None,
    pct_cols: list[str] | None = None,
    pct_scale100: bool = False,
    col_formats: dict[str, object] | None = None,
    **_unused,
):
    # Bảng học thuật kiểu booktabs tối giản: không thanh màu nền, không nút sắp xếp, an toàn HTML 100%.
    out = df.copy()
    int_set = set(int_cols or [])
    pct_set = set(pct_cols or [])
    custom_fmt = dict(col_formats or {})

    num_cols: set[str] = set()
    for c in out.columns:
        if c in int_set or c in pct_set or pd.api.types.is_numeric_dtype(out[c]):
            num_cols.add(c)

    parts = ['<div class="ah-table-scroll"><table class="dataframe ah-clean-table">']
    if caption:
        parts.append(f"<caption>{html.escape(str(caption))}</caption>")

    parts.append("<thead><tr>")
    for col in out.columns:
        cls = ' class="num-col"' if col in num_cols else ""
        parts.append(f"<th{cls}>{html.escape(str(col))}</th>")
    parts.append("</tr></thead><tbody>")

    for _, row in out.iterrows():
        parts.append("<tr>")
        for col in out.columns:
            val = row[col]
            if col in custom_fmt:
                txt = custom_fmt[col](val)
            elif isinstance(val, (list, tuple)):
                txt = ", ".join(str(x) for x in val) if val else "-"
            elif val is None or (not isinstance(val, (str, bytes)) and pd.isna(val)):
                txt = "-"
            elif col in int_set or pd.api.types.is_integer_dtype(out[col]):
                txt = fmt_int(val)
            elif col in pct_set:
                txt = fmt_pct(val, precision, pct_scale100)
            elif pd.api.types.is_float_dtype(out[col]):
                txt = fmt_num(val, precision)
            else:
                txt = str(val)
            cls = ' class="num-col"' if col in num_cols else ""
            esc = html.escape(str(txt))
            if not cls and str(txt).startswith(("http://", "https://", "10.")):
                esc = f'<span class="ah-breakable">{esc}</span>'
            parts.append(f"<td{cls}>{esc}</td>")
        parts.append("</tr>")

    parts.append("</tbody></table></div>")
    return HTML("".join(parts))


def draw_pipeline_diagram(title: str, stages: list[tuple[str, str]]) -> None:
    # Sơ đồ khối quy trình tối giản chuẩn báo cáo khoa học.
    fig, ax = plt.subplots(figsize=(10.0, 2.05))
    ax.set_xlim(0, 10.0)
    ax.set_ylim(0, 2.05)
    ax.axis("off")
    n = len(stages)
    bw = 2.22
    gap = (9.70 - n * bw) / max(n - 1, 1)
    for idx, (st_title, st_desc) in enumerate(stages):
        x0 = 0.15 + idx * (bw + gap)
        ax.add_patch(Rectangle((x0, 0.18), bw, 1.56, facecolor="#FFFFFF", edgecolor="#CBD5E1", linewidth=1.0))
        ax.add_patch(Rectangle((x0, 1.32), bw, 0.42, facecolor="#EDF2F7", edgecolor="#CBD5E1", linewidth=0.8))
        ax.add_patch(Rectangle((x0, 1.32), 0.06, 0.42, facecolor="#1B365D", edgecolor="none"))
        ax.text(x0 + bw / 2 + 0.02, 1.53, st_title, ha="center", va="center", color="#1B365D", fontsize=8.8, fontweight="bold")
        ax.text(x0 + bw / 2, 0.74, st_desc, ha="center", va="center", color="#334155", fontsize=8.0, linespacing=1.35)
        if idx < n - 1:
            ax.add_patch(FancyArrowPatch(
                (x0 + bw + 0.04, 0.96),
                (x0 + bw + gap - 0.04, 0.96),
                arrowstyle="-|>,head_length=5,head_width=3.2",
                color="#64748B",
                lw=1.1,
            ))
    ax.set_title(title, pad=4, fontsize=10.0, fontweight="bold", color="#1B365D")
    plt.tight_layout()
    plt.show()

In [ ]:
ps = m1["performance_summary"]
df_7m = pd.DataFrame([
    ("1. Thời gian xử lý 1 bài đơn lẻ (1 phiên bản, tuần tự lạnh / khấu hao lô)",
     f"{fmt_num(ps['single_paper_cold_1ver_sec'], 2)}s (lạnh) | {fmt_num(ps['single_paper_amortized_1ver_sec'], 2)}s (khấu hao lô)",
     "Đo đơn bài thực tế trên Colab CPU đạt trung vị p50 = 3,08 - 4,82s (sát giới hạn nghỉ 3,0s của arXiv)"),
    ("2. Thời gian trung bình mỗi bài báo trên tập yêu cầu",
     f"{fmt_num(ps['full_4shards_avg_sec_per_paper'], 4)}s/bài (4 phiên đầy đủ) | {fmt_num(ps['all_5members_avg_sec_per_paper'], 4)}s/bài (đủ 25.000 bài)",
     f"Tổng thời gian thực 6 phiên = {fmt_num(ps['all_5members_wall_sec'], 1)}s ({fmt_num(ps['all_5members_wall_sec']/3600, 2)} giờ, trung bình {fmt_num(ps['all_5members_avg_sec_per_version'], 2)}s/phiên bản)"),
    ("3. Tổng thời gian khám phá danh mục (Entry Discovery)",
     f"{fmt_num(ps['discovery_4shards_sec'], 2)}s (4 phiên đầy đủ) | {fmt_num(ps['discovery_measured_total_sec'], 2)}s (cộng phiên TRI-resume)",
     f"{fmt_int(ps['discovery_4shards_calls'])} lô Atom (~0,105s/bài, chiếm 1,30% tổng thời gian) và {fmt_num(ps['id_boundary_tri_resume_sec'], 2)}s dò biên tháng"),
    ("4. Bộ nhớ RAM cực đại (Peak RSS)",
     f"{fmt_num(ps['peak_rss_mib'], 2)} MiB (OS ru_maxrss: {fmt_num(ps['os_reported_peak_rss_mib'], 2)} MiB)",
     "Chiếm 9,46% trên tổng dung lượng 12.975,52 MiB RAM của máy ảo Google Colab CPU-only"),
    ("5. Bộ nhớ RAM trung bình gia quyền theo thời gian (Weighted Avg RSS)",
     f"{fmt_num(ps['weighted_avg_rss_mib'], 2)} MiB",
     f"Tính trên {fmt_int(ps['rss_total_samples_4shards'])} mẫu chu kỳ 0,5s (phiên TRI-resume đạt trung bình 625,73 MiB trên {fmt_int(ps['rss_samples_tri_resume'])} mẫu)"),
    ("6. Dung lượng đĩa cực đại yêu cầu trên 1 phiên Colab (Peak Disk)",
     f"{fmt_num(ps['tri_resume_sampled_peak_disk_bytes']/1024**3, 2)} - {fmt_num(ps['max_member_end_disk_bytes']/1024**3, 2)} GiB",
     f"Đỉnh lấy mẫu phiên TRI-resume: {fmt_int(ps['tri_resume_sampled_peak_disk_bytes'])} B; cuối phiên lớn nhất (TRIET): {fmt_int(ps['max_member_end_disk_bytes'])} B"),
    ("7. Dung lượng lưu trữ đầu ra cuối cùng (Final Output Size)",
     f"Nguồn TeX/Bib sạch: {fmt_num(ps['final_clean_corpus_bytes']/1024**3, 2)} GiB | Bảng chuẩn M1: {fmt_num(ps['final_course_export_total_bytes']/1048576, 2)} MiB",
     f"Working Parquet hợp nhất: {fmt_num(ps['final_merged_working_parquet_bytes']/1048576, 2)} MiB; papers (3,52 MiB) + references (74,23 MiB) = {fmt_int(ps['final_course_export_total_bytes'])} B"),
], columns=["Chỉ số hiệu năng bắt buộc (M1)", "Giá trị đo trên Colab CPU-only", "Chi tiết kỹ thuật"])
display(style_table(df_7m, caption="Bảng 5.1. Tổng hợp 7 chỉ số hiệu năng bắt buộc của trình thu thập Milestone 1 trên Google Colab CPU-only."))

In [ ]:
sess_m1 = pd.DataFrame(m1["colab_sessions_full"])
view_s1 = pd.DataFrame({
    "Phiên chạy": sess_m1["session_label"],
    "Xử lý": sess_m1["processed_this_run"],
    "Lũy kế": sess_m1["completed_in_checkpoint"],
    "Thời gian (h)": sess_m1["runtime_sec"] / 3600.0,
    "Giây/bài": sess_m1["avg_sec_per_processed"],
    "Peak RAM (MiB)": sess_m1["peak_rss_mib"],
    "Mean RAM (MiB)": sess_m1["avg_rss_mib"],
    "Đĩa cuối (GiB)": sess_m1["end_disk_bytes"] / (1024**3),
    "Lý do dừng": sess_m1["stop_reason"],
})
display(style_table(
    view_s1,
    caption="Bảng 5.2. Thông số vận hành của 6 phiên Google Colab CPU-only ở giai đoạn Milestone 1.",
    int_cols=["Xử lý", "Lũy kế"],
))

ph_raw = m1["phases_aggregated"]
df_ph = pd.DataFrame([
    {
        "Pha xử lý M1": str(k),
        "Số lượt gọi": int(v["calls"]),
        "Tổng giây cộng dồn": float(v["elapsed_sec"]),
        "Tổng giờ": float(v["elapsed_sec"]) / 3600.0,
        "TB giây / lượt": float(v["elapsed_sec"]) / max(int(v["calls"]), 1),
    }
    for k, v in ph_raw.items()
]).sort_values("Tổng giây cộng dồn", ascending=False)
display(style_table(
    df_ph,
    caption="Bảng 5.3. Phân rã thời gian 12 pha xử lý M1 trên 4 phiên đầy đủ (tổng các pha gấp 1,115 lần thời gian thực nhờ chạy đa luồng gối đầu).",
    int_cols=["Số lượt gọi"],
))

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10.4, 3.8))
xs = np.arange(len(sess_m1))
ax1.bar(xs, sess_m1["avg_sec_per_processed"].to_numpy(dtype=float), color=BOOK_PALETTE["navy"], width=0.54)
ax1.axhline(ps["all_5members_avg_sec_per_paper"], color=BOOK_PALETTE["slate"], ls="--", lw=1.2,
            label=f"Trung bình toàn nhóm ({fmt_num(ps['all_5members_avg_sec_per_paper'], 2)}s/bài)")
ax1.set_xticks(xs)
ax1.set_xticklabels([str(v) for v in sess_m1["session_label"]], rotation=20, ha="right")
ax1.set_ylim(0, 11.5)
ax1.grid(axis="y")
ax1.set_ylabel("Giây / bài xử lý")
ax1.set_title("Hình 5.1a. Tốc độ trung bình từng phiên M1")
ax1.legend(fontsize=8.0)
for idx_s, v_s in enumerate(sess_m1["avg_sec_per_processed"]):
    ax1.text(idx_s, float(v_s) + 0.22, f"{fmt_num(v_s, 2)}s", ha="center", fontsize=7.8)

mem_s = sess_m1.dropna(subset=["peak_rss_mib"]).reset_index(drop=True)
xm = np.arange(len(mem_s))
wm = 0.34
ax2.bar(xm - wm / 2, mem_s["avg_rss_mib"].to_numpy(dtype=float), wm, color=BOOK_PALETTE["steel"], label="Mean RSS (MiB)")
ax2.bar(xm + wm / 2, mem_s["peak_rss_mib"].to_numpy(dtype=float), wm, color=BOOK_PALETTE["navy"], label="Peak RSS (MiB)")
ax2.set_xticks(xm)
ax2.set_xticklabels([str(v) for v in mem_s["session_label"]], rotation=20, ha="right")
ax2.set_ylim(0, 1480)
ax2.grid(axis="y")
ax2.set_ylabel("Bộ nhớ RAM (MiB)")
ax2.set_title("Hình 5.1b. Tiêu thụ RAM trên Colab CPU (< 10% trần 12,7 GiB)")
ax2.legend(fontsize=8.0)
for idx_m, r_m in mem_s.iterrows():
    ax2.text(idx_m + wm / 2, float(r_m["peak_rss_mib"]) + 25, fmt_num(r_m["peak_rss_mib"], 0), ha="center", fontsize=7.8)
plt.tight_layout()
plt.show()

fig, ax = plt.subplots(figsize=(9.8, 3.9))
ph_plot = df_ph.sort_values("Tổng giây cộng dồn", ascending=True)
yp = np.arange(len(ph_plot))
cols_p = [BOOK_PALETTE["steel"] if p == "discovery" else BOOK_PALETTE["navy"] for p in ph_plot["Pha xử lý M1"]]
ax.barh(yp, ph_plot["Tổng giây cộng dồn"].to_numpy(dtype=float), color=cols_p, height=0.58)
ax.set_yticks(yp)
ax.set_yticklabels([str(v) for v in ph_plot["Pha xử lý M1"]])
ax.set_xscale("log")
ax.set_xlim(0.18, 6e5)
ax.grid(axis="x")
ax.set_xlabel("Tổng giây cộng dồn (thang log10)")
ax.set_title("Hình 5.2. Thời gian cộng dồn của 12 pha xử lý M1 (Discovery chiếm 2.094,9s)")
for idx_p, r_p in enumerate(ph_plot.to_dict("records")):
    ax.text(max(float(r_p["Tổng giây cộng dồn"]), 0.22) * 1.18, idx_p,
            f"{fmt_num(r_p['Tổng giây cộng dồn'], 1)}s ({fmt_int(r_p['Số lượt gọi'])} lượt)",
            va="center", fontsize=7.9)
plt.tight_layout()
plt.show()

## Tài liệu và mã nguồn đối chiếu

- [Đặc tả chính thức Milestone 1 - Data Crawling](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/crawling/).
- [arXiv API User's Manual](https://info.arxiv.org/help/api/user-manual.html), [API Terms of Use](https://info.arxiv.org/help/api/tou.html) và [OAI-PMH Cutover (`oaipmh.arxiv.org`)](https://info.arxiv.org/help/oa/index.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã nguồn triển khai: [`src/crawler/profiler.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/profiler.py).
- Mã nguồn triển khai: [`src/crawler/benchmark_sessions.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/src/crawler/benchmark_sessions.py).
- Mã nguồn triển khai: [`tools/m1_colab_tri_resume.py`](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/main/tools/m1_colab_tri_resume.py).